# Hybrid GRU-ESN Models For Stock Prediction: Preliminary Research Report

In [1]:
import os
import sys
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score

from src.my_engine.financial_data import make_single_stock_df
from src.my_engine.config import ModelConfig, TrainerConfig, MetricsConfig
from src.my_engine.utils import build_model, make_optimizer, get_direction_accuracy
from src.my_engine.trainer import Trainer, RidgeRegressionTrainer
from src.my_engine.data import get_dataloaders

In [38]:
# Define paths to config files for later
gru_msft_config_path = "../best_model_archs/gru_msft.pt"
gru_esn_msft_config_path = "../best_model_archs/deep_esn_gru_msft.pt"
gru_esn_nvda_config_path = "../best_model_archs/nvda_deep_esn_gru.pt"
gru_esn_qcom_config_path = "../best_model_archs/qcom_deep_gru_esn.pt"
esn_msft_config_path = "../best_model_archs/esn_msft.pt"
esn_nvda_config_path = "../best_model_archs/esn_nvda.pt"
esn_qcom_config_path = "../best_model_archs/qcom_esn.pt"

In [3]:
# Define our datasets
train_msft_ds, val_msft_ds, test_msft_ds, msft_scaler, features = make_single_stock_df("MSFT", "5y")
train_nvda_ds, val_nvda_ds, test_nvda_ds, nvda_scaler, features = make_single_stock_df("NVDA", "5y")
train_qcom_ds, val_qcom_ds, test_qcom_ds, qcom_scaler, features = make_single_stock_df("QCOM", "5y")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"


## Introduction
This report investigates the performance of several neural network architectures for stock market prediction, with a particular focus on a novel hybrid model combining Gated Recurrent Units (GRUs) and Echo State
Networks (ESNs). Financial time series data presents a challenging modeling task due to its inherent noise, non-stationarity, and low signal-to-noise ratio, making it a suitable testbed for evaluating model
robustness and generalization. Motivated by the complementary strengths of GRUs, which learn adaptive temporal dependencies, and ESNs, which provide rich dynamical representations through fixed reservoirs and act like
a memory bank, this work proposes and analyzes a GRU-ESN hybrid architecture. The goal is to assess whether this combined approach can improve performance on directional stock prediction relative to traditional models.


## Model Architectures
We are now going to be taking a brief look at each of the architectures that we are going to explore
### GRU
We will just touch briefly on GRUs. As you most likely already know, GRUs are a common type of RNN that has a hidden state that acts as a gradient highway to allow the model to learn long-range dependencies. GRUs have
two learned gates to an update gate to update the hidden state and the reset gate that removes no longer necessary data from the hidden state.
### ESN
ESNs or echo-state networks are a type of reservoir computing model and are different from traditional neural networks. Reservoir computing models work by randomly initializing a reservoir of neurons and also the connections
between those neurons and the weights of inputs into these neurons. These weights are frozen so they are not updated during training only the output weights are trained, but they are traditionally trained with ridge regression
instead of backprop like a regular neural network. These models require a lot of fine-tuning to find good parameters for reservoir initialization and also a bit of luck since the reservoirs are randomly initialized.
### GRU-ESNs
This is a new novel architecture of my own creation. This is a hybrid model that combines GRUs and ESNs into one model where the state from each time step for each layer of the ESN is added into the hidden state of the GRU
using a new learned gate the `esn_gate`. This allows the model to learn when it is useful to use the output from the ESN state.

## Data and Feature Engineering
### Data
The data that used for this project was from the `yfinance` python package. We used this package to retrieve
data for three different companies' stock prices, Microsoft (MSFT), Nvidia (NVDA), and Qualcomm (QCOM).
For each of these companies the last 5 years of data was retrieved and daily data was used.
### Feature Engineering
To capture meaningful patterns in stock price dynamics, a set of engineered features was constructed from raw OHLCV (Open, High, Low, Close, Volume) data. These features are designed to represent short-term
momentum, volatility, trading activity, and intraday behavior, all of which are commonly used in financial time series modeling. Below are the features that were created
- `log_returns` - the log returns for the day and our used as our target but it is the next days log return
- `open_close` - the difference between open and close divided by close
- `overnight_gap` - the change in price over night
- `ret_1` - the one day return
- `ret_5` - the 5 day return
- `log_volume` - the log volume for the day
- `log_intraday_chng` - log of high divided by low capturing price change during the day
- `variance` - the variance in the one day returns
- 10 20 and 50  day log return moving averages
### Preprocessing
After creating the features the data is split into training validation and test sets. Then a standard
scaler is fit on the training data and then all the data is scaled using this scaler. Next each split is converted into a sliding window dataset with a 30 day window. The `log_return` of the day after the end of the window
is the target for the model. Then this data is converted into a PyTorch dataset.

## Experimental Setup
### Evaluation Metics
The only evaluation metric we are going to be using is directional accuracy because the MSE for all the models is fairly large. However we train using `log_returns` we did this because when training with direction as the
target the models mostly converged to either only up or down. The baseline for accuracy will be always predicting upward price movement.
### Training
To train all of these models extensive hyperparameter sweeps were used over all parameters for the
models. For GRU we swept over hidden_size, num_layers, learning_rate, etc. For the ESN we swept over
spectral_radius, reservoir_size, reservoir_sparsity, etc. And for our hybrid we swept over all of these
different parameters since this new model used most of them.
### Baselines
The baseline for directional accuracy is always predicting upward price movement. For MSE we will use
a model that just predicts no returns for the day.

## Model Performance
Now we are getting into the fun stuff and actually looking at how the models performed. Since these are only preliminary results some of the model types were not trained on all the stocks, specifically GRU because it
had inferior performance from the get go. We are going to be using all the best models that were found from sweeps in the below results. We are going to do this by loading in the model architecture from the best models and
training them below to see the performance.
## GRU
As already stated the GRU was not the best model it achieved much lower directionally accuracy on MSFT then the both other types of models.

In [5]:
msft_gru_config = torch.load(gru_msft_config_path)

deep_gru_config = ModelConfig(
    model_type=msft_gru_config['model_type'],
    hidden_units=msft_gru_config['hidden_units'],
    rnn_hidden_size=msft_gru_config['rnn_hidden_size'],
    rnn_num_layers=msft_gru_config['rnn_num_layers'],
    dropout=msft_gru_config['dropout']
)

best_gru_model = build_model(input_spec=len(features), config=deep_gru_config, num_outputs=1)

gru_best_trainer_config = TrainerConfig(
    optimizer_name=msft_gru_config['optimizer_name'],
    learning_rate=msft_gru_config['learning_rate'],
    weight_decay=msft_gru_config['weight_decay'],
    use_scheduler=msft_gru_config['use_scheduler'],
    early_stopping_min_delta=msft_gru_config['early_stopping_min_delta'],
    early_stopping_patience=None,
    trainer_batch_size=msft_gru_config['trainer_batch_size'],
    evaluator_batch_size=msft_gru_config['evaluator_batch_size'],
    num_epochs=50,
    device=device,
)

train_loader, test_loader, _ = get_dataloaders(
    train_ds=train_msft_ds,
    eval_ds=test_msft_ds,
    train_batch_size=gru_best_trainer_config.trainer_batch_size,
    eval_batch_size=gru_best_trainer_config.evaluator_batch_size,
    time_series=True,
)

with Trainer(
    model=best_gru_model,
    optimizer=make_optimizer(best_gru_model.parameters(), gru_best_trainer_config),
    criterion=torch.nn.MSELoss(),
    config=gru_best_trainer_config,
    metrics_config=MetricsConfig(task="regression", names=["mse", "mae"])
) as trainer:
    best_results = trainer.fit(train_loader, test_loader)

Epoch 0: Train Loss=1.0267,Val Loss=1.4152, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0267, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7406, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.4152, device='cuda:0'), 'MeanAbsoluteError': tensor(0.8163, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=1.0257,Val Loss=1.4154, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0257, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7402, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.4154, device='cuda:0'), 'MeanAbsoluteError': tensor(0.8164, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=1.0252,Val Loss=1.4156, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0252, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7399, device='cuda:0')}, Test: {'MeanSquaredErr

In [6]:
best_weights = torch.load("checkpoints/best.pt", weights_only=True)
best_gru_model.load_state_dict(best_weights['model_state_dict'])

<All keys matched successfully>

In [7]:
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(best_gru_model, test_loader, msft_scaler, features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Directional accuracy: 54.44%
Directional precision: 54.44%
Baseline accuracy: 54.44%


As you can see from the above the GRU does not perform very well at all its directional accuracy is less than baseline meaning that this is a very poor model. For these reasons we decided to not keep testing with this
model and only focus on the hybrid and esn models.

### ESN
The ESNs performed very well on most of the data that we through at it. However you do sometimes need to run the model a few times to get a good model. This is do to the random initialization of the weights. If
you run the below cells a few times you will be able to see this in action. We are going to load and run all of the ESN models and then discuss thier performance.

In [14]:
# Load all the config files
msft_esn_config = torch.load(esn_msft_config_path)
qcom_esn_config = torch.load(esn_nvda_config_path)
nvda_esn_config = torch.load(esn_qcom_config_path)

#### Microsoft

In [36]:
best_esn_config = ModelConfig(
    model_type=msft_esn_config['model_type'],
    leak_rate=msft_esn_config['leak_rate'],
    spectral_radius=msft_esn_config['spectral_radius'],
    reservoir_size=msft_esn_config['reservoir_size'],
    reservoir_sparsity=msft_esn_config['reservoir_sparsity'],
    hidden_units=msft_esn_config['hidden_units'],
    input_scale=msft_esn_config['input_scale'],
)

msft_esn_model = build_model(input_spec=len(features), config=best_esn_config, num_outputs=1)

best_esn_trainer_config = TrainerConfig(
    num_epochs=50,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_msft_ds,
    eval_ds=val_msft_ds,
    test_ds=test_msft_ds,
    train_batch_size=best_esn_trainer_config.trainer_batch_size,
    eval_batch_size=best_esn_trainer_config.evaluator_batch_size,
    time_series=True,
)

trainer = RidgeRegressionTrainer(
    model=msft_esn_model,
    criterion=torch.nn.MSELoss(),
    config=best_esn_trainer_config,
    ridge_alpha=msft_esn_config['ridge_alpha'],
)
best_results = trainer.fit(train_loader, val_loader)
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(msft_esn_model, test_loader, msft_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Directional accuracy: 60.00%
Directional precision: 63.27%
Baseline accuracy: 54.44%


#### Nvidia

In [58]:
best_esn_config = ModelConfig(
    model_type=nvda_esn_config['model_type'],
    leak_rate=nvda_esn_config['leak_rate'],
    spectral_radius=nvda_esn_config['spectral_radius'],
    reservoir_size=nvda_esn_config['reservoir_size'],
    reservoir_sparsity=nvda_esn_config['reservoir_sparsity'],
    hidden_units=nvda_esn_config['hidden_units'],
    input_scale=nvda_esn_config['input_scale'],
)

nvda_esn_model = build_model(input_spec=len(features), config=best_esn_config, num_outputs=1)

best_esn_trainer_config = TrainerConfig(
    num_epochs=50,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_nvda_ds,
    eval_ds=val_nvda_ds,
    test_ds=test_nvda_ds,
    train_batch_size=best_esn_trainer_config.trainer_batch_size,
    eval_batch_size=best_esn_trainer_config.evaluator_batch_size,
    time_series=True,
)

trainer = RidgeRegressionTrainer(
    model=nvda_esn_model,
    criterion=torch.nn.MSELoss(),
    config=best_esn_trainer_config,
    ridge_alpha=nvda_esn_config['ridge_alpha'],
)
best_results = trainer.fit(train_loader, val_loader)
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(nvda_esn_model, test_loader, nvda_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Directional accuracy: 57.78%
Directional precision: 58.33%
Baseline accuracy: 46.67%


#### Qualcomm

In [64]:
best_esn_config = ModelConfig(
    model_type=qcom_esn_config['model_type'],
    leak_rate=qcom_esn_config['leak_rate'],
    spectral_radius=qcom_esn_config['spectral_radius'],
    reservoir_size=qcom_esn_config['reservoir_size'],
    reservoir_sparsity=qcom_esn_config['reservoir_sparsity'],
    hidden_units=qcom_esn_config['hidden_units'],
    input_scale=qcom_esn_config['input_scale'],
)

qcom_esn_model = build_model(input_spec=len(features), config=best_esn_config, num_outputs=1)

best_esn_trainer_config = TrainerConfig(
    num_epochs=50,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_qcom_ds,
    eval_ds=val_qcom_ds,
    test_ds=test_qcom_ds,
    train_batch_size=best_esn_trainer_config.trainer_batch_size,
    eval_batch_size=best_esn_trainer_config.evaluator_batch_size,
    time_series=True,
)

trainer = RidgeRegressionTrainer(
    model=qcom_esn_model,
    criterion=torch.nn.MSELoss(),
    config=best_esn_trainer_config,
    ridge_alpha=qcom_esn_config['ridge_alpha'],
)
best_results = trainer.fit(train_loader, val_loader)
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(qcom_esn_model, test_loader, qcom_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Directional accuracy: 56.67%
Directional precision: 61.29%
Baseline accuracy: 51.11%


As you can see from looking at the results above these models perform fairly well in directional accuracy with the deep ESNs achieving better accuracy than the baseline in all cases. In the case of Microsoft and Qualcomm
the model achieves over 60% accuraccy which is very high for price movement prediction meaning that these models could potentially be used for training. However like I mentioned above you need to run these models
multiple times to get good outputs, I had to do this to get the performance numbers above. However this is not really an inconvience since these models traing extremely fast through ridge regression

### Hybrid GRU ESN
Now it's time for the really fun stuff. I am now going to go a bit more indepth into how the DeepESNGRU model that I created works. Firstly we are only going to be focusing on this model because from my testing this model
performed the best form the other different variations that were created. So how this model works is that you have a regular GRU cell except now each GRU cell has its ESN attached to it. This ESN is slightly different
because now the output weights from the ESN are being learned by the ESN gate so they are not optimized by ridge regression anymore. Now the equation for the esn gate is
`h_new = (1.0 - esn_gate) * h_gru + esn_gate * projected_r_t` where `esn_gate` is the output vector from the esn gate and decides what should be taken out of the gru hidden state and replaced with the ESN state `projected_r_t`
which is the learned output of the ESN at that layer. Now lets look at the best models that we found and their performance.

In [39]:
msft_esn_gru_config = torch.load(gru_esn_msft_config_path)
nvda_esn_gru_config = torch.load(gru_esn_nvda_config_path)
qcom_esn_gru_config = torch.load(gru_esn_qcom_config_path)

#### Microsoft

In [67]:
best_msft_config = ModelConfig(
    model_type=msft_esn_gru_config['model_type'],
    leak_rate=msft_esn_gru_config['leak_rate'],
    spectral_radius=msft_esn_gru_config['spectral_radius'],
    reservoir_size=msft_esn_gru_config['reservoir_size'],
    reservoir_sparsity=msft_esn_gru_config['reservoir_sparsity'],
    hidden_units=msft_esn_gru_config['hidden_units'],
    input_scale=msft_esn_gru_config['input_scale'],
    rnn_hidden_size=msft_esn_gru_config['rnn_hidden_size']
)

best_msft_model = build_model(input_spec=len(features), config=best_msft_config, num_outputs=1)

best_msft_trainer_config = TrainerConfig(
    optimizer_name=msft_esn_gru_config['optimizer_name'],
    learning_rate=msft_esn_gru_config['learning_rate'],
    weight_decay=msft_esn_gru_config['weight_decay'],
    use_scheduler=msft_esn_gru_config['use_scheduler'],
    early_stopping_min_delta=msft_esn_gru_config['early_stopping_min_delta'],
    early_stopping_patience=None,
    trainer_batch_size=msft_esn_gru_config['trainer_batch_size'],
    evaluator_batch_size=msft_esn_gru_config['evaluator_batch_size'],
    num_epochs=25,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_msft_ds,
    eval_ds=val_msft_ds,
    test_ds=test_msft_ds,
    train_batch_size=best_msft_trainer_config.trainer_batch_size,
    eval_batch_size=best_msft_trainer_config.evaluator_batch_size,
    time_series=True,
)

with Trainer(
        model=best_msft_model,
        optimizer=make_optimizer(filter(lambda p: p.requires_grad, best_msft_model.parameters()),
                                 best_msft_trainer_config),
        criterion=torch.nn.MSELoss(),
        config=best_msft_trainer_config,
        metrics_config=MetricsConfig(task="regression", names=["mse", "mae"])
) as trainer:
    best_results = trainer.fit(train_loader, test_loader)
best_weights = torch.load("checkpoints/best.pt", weights_only=True)
best_msft_model.load_state_dict(best_weights['model_state_dict'])
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(best_msft_model, test_loader, msft_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Epoch 0: Train Loss=1.0351,Val Loss=1.4476, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0351, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7399, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.4476, device='cuda:0'), 'MeanAbsoluteError': tensor(0.8301, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=1.0368,Val Loss=1.4724, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0368, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7472, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.4724, device='cuda:0'), 'MeanAbsoluteError': tensor(0.8426, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=1.0284,Val Loss=1.4385, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0284, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7436, device='cuda:0')}, Test: {'MeanSquaredErr

#### Nvidia

In [48]:
best_nvda_config = ModelConfig(
    model_type=nvda_esn_gru_config['model_type'],
    leak_rate=nvda_esn_gru_config['leak_rate'],
    spectral_radius=nvda_esn_gru_config['spectral_radius'],
    reservoir_size=nvda_esn_gru_config['reservoir_size'],
    reservoir_sparsity=nvda_esn_gru_config['reservoir_sparsity'],
    hidden_units=nvda_esn_gru_config['hidden_units'],
    input_scale=nvda_esn_gru_config['input_scale'],
    rnn_hidden_size=nvda_esn_gru_config['rnn_hidden_size']
)

best_nvda_model = build_model(input_spec=len(features), config=best_nvda_config, num_outputs=1)

best_nvda_trainer_config = TrainerConfig(
    optimizer_name=nvda_esn_gru_config['optimizer_name'],
    learning_rate=nvda_esn_gru_config['learning_rate'],
    weight_decay=nvda_esn_gru_config['weight_decay'],
    use_scheduler=nvda_esn_gru_config['use_scheduler'],
    early_stopping_min_delta=nvda_esn_gru_config['early_stopping_min_delta'],
    early_stopping_patience=None,
    trainer_batch_size=nvda_esn_gru_config['trainer_batch_size'],
    evaluator_batch_size=nvda_esn_gru_config['evaluator_batch_size'],
    num_epochs=25,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_nvda_ds,
    eval_ds=val_nvda_ds,
    test_ds=test_nvda_ds,
    train_batch_size=best_nvda_trainer_config.trainer_batch_size,
    eval_batch_size=best_nvda_trainer_config.evaluator_batch_size,
    time_series=True,
)

with Trainer(
        model=best_nvda_model,
        optimizer=make_optimizer(filter(lambda p: p.requires_grad, best_nvda_model.parameters()),
                                 best_nvda_trainer_config),
        criterion=torch.nn.MSELoss(),
        config=best_nvda_trainer_config,
        metrics_config=MetricsConfig(task="regression", names=["mse", "mae"])
) as trainer:
    best_results = trainer.fit(train_loader, test_loader)
best_weights = torch.load("checkpoints/best.pt", weights_only=True)
best_nvda_model.load_state_dict(best_weights['model_state_dict'])
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(best_nvda_model, test_loader, nvda_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Epoch 0: Train Loss=1.0262,Val Loss=0.4033, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0262, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7575, device='cuda:0')}, Test: {'MeanSquaredError': tensor(0.4033, device='cuda:0'), 'MeanAbsoluteError': tensor(0.4893, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=1.0154,Val Loss=0.4002, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0154, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7532, device='cuda:0')}, Test: {'MeanSquaredError': tensor(0.4002, device='cuda:0'), 'MeanAbsoluteError': tensor(0.4874, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=1.0071,Val Loss=0.3983, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError':

#### Qualcomm

In [46]:
best_qcom_config = ModelConfig(
    model_type=qcom_esn_gru_config['model_type'],
    leak_rate=qcom_esn_gru_config['leak_rate'],
    spectral_radius=qcom_esn_gru_config['spectral_radius'],
    reservoir_size=qcom_esn_gru_config['reservoir_size'],
    reservoir_sparsity=qcom_esn_gru_config['reservoir_sparsity'],
    hidden_units=qcom_esn_gru_config['hidden_units'],
    input_scale=qcom_esn_gru_config['input_scale'],
    rnn_hidden_size=qcom_esn_gru_config['rnn_hidden_size']
)

best_qcom_model = build_model(input_spec=len(features), config=best_qcom_config, num_outputs=1)

best_qcom_trainer_config = TrainerConfig(
    optimizer_name=qcom_esn_gru_config['optimizer_name'],
    learning_rate=qcom_esn_gru_config['learning_rate'],
    weight_decay=qcom_esn_gru_config['weight_decay'],
    use_scheduler=qcom_esn_gru_config['use_scheduler'],
    early_stopping_min_delta=qcom_esn_gru_config['early_stopping_min_delta'],
    early_stopping_patience=None,
    trainer_batch_size=qcom_esn_gru_config['trainer_batch_size'],
    evaluator_batch_size=qcom_esn_gru_config['evaluator_batch_size'],
    num_epochs=25,
    device=device,
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_ds=train_qcom_ds,
    eval_ds=val_qcom_ds,
    test_ds=test_qcom_ds,
    train_batch_size=best_qcom_trainer_config.trainer_batch_size,
    eval_batch_size=best_qcom_trainer_config.evaluator_batch_size,
    time_series=True,
)

with Trainer(
        model=best_qcom_model,
        optimizer=make_optimizer(filter(lambda p: p.requires_grad, best_qcom_model.parameters()),
                                 best_qcom_trainer_config),
        criterion=torch.nn.MSELoss(),
        config=best_qcom_trainer_config,
        metrics_config=MetricsConfig(task="regression", names=["mse", "mae"])
) as trainer:
    best_results = trainer.fit(train_loader, test_loader)
best_weights = torch.load("checkpoints/best.pt", weights_only=True)
best_qcom_model.load_state_dict(best_weights['model_state_dict'])
deep_best_dir_acc, true_dirs, pred_dirs = get_direction_accuracy(best_qcom_model, test_loader, qcom_scaler,
                                                                 features.index('log_returns'))
precision = precision_score(true_dirs, pred_dirs)
print(f"Directional accuracy: {deep_best_dir_acc:.2%}")
print(f"Directional precision: {precision:.2%}")
print(f"Baseline accuracy: {true_dirs.mean():.2%}")

Epoch 0: Train Loss=1.0231,Val Loss=1.0506, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0231, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7353, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.0506, device='cuda:0'), 'MeanAbsoluteError': tensor(0.6328, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=1.0163,Val Loss=1.0470, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0163, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7336, device='cuda:0')}, Test: {'MeanSquaredError': tensor(1.0470, device='cuda:0'), 'MeanAbsoluteError': tensor(0.6321, device='cuda:0')}
--> Saving checkpoint: ./checkpoints/last.pt
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=1.0127,Val Loss=1.0441, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError':

From looking at the results above we can see that the hybrid model gets pretty similar performance but maybe slightly worse than the ESNs and MSFT is below the baseline. However it is hard to tell since these models also suffer from the random initialization of the
ESNs and their performance heavily depends upon these. In testing in different notebooks I was getting similar if not slightly better performance with these hybrid models. This is one potentially downside of them they
can sometimes get slightly better performance but they take much longer to train so it is harder to get there then just with regular ESNs.

## Performance Comparison and Conclusion
For the performance comparison we are going to leave out the plain GRU model since it is so far behind and it being so far behind indicates that adding the ESN into the GRU model improves its performance. Now were
are going to compare the performance for each differrent stock. However all of these should be taken with a grain of salt since the random initialization of the ESN can cause the performance to vary.
### MSFT
Here the ESN beats out the hybrid model in accuracy with 60% versus the 53.33% for the hybrid model. The ESN also beats out the hybrid in directional precision with a precision of 63.27% versus 56.6%. This result should
be taken with a grain of salt since the original hybrid sweep was done with slightly different features. I still used this model as I did not have time to rerun it but that is a future improvement that I want to make.
In my original testing this model was achieving 58% accuracy.
### NVDA
The ESN outperfroms the hybrid model in both accuracy and precision with an accuracy of 57.78% and a directional precision of 58.33% versus the hybrid model's accuracy of 50.00% and directional precision of 45.45%. The hybrid
model really seemed to struggle with NVDA however in previous testing it seemed to perform ok.
### QCOM
From looking at the accuracy numbers the ESN outperforms the hybrid model slightly with an accuracy of 56.67% while the hybrid model has an accuracy of 55.56%. However the hybrid model has a much better directional precision when compared
to the ESN with a directional precision of 71.43% versus the ESN's 61.29%. Indicating that the hybrid model may be better because it is right more of the time it predicts upward movement.
### Conclusion
Overall we can see that the ESNs normally outperform in accuracy but they sometimes have much worse precision than the hybrid models. Also all of these results should not be taken to seriously due to the random nature of
these models. During running of the above code cells there were times when the ESN was much worse than the hybrid model so this would require much further testing to draw any concrete conclusions.

## Future Work
There are many improvements that I would like to make to the hybrid models and experiments going into the future. First I would like to explore more features for the different tickers as I know that this can greatly increase
performance and would like to see if certain features would help the hybrid model. Also I would like to update the DeepESNGRU to actually use a DeepESN where the internal state is extracted for each layer of the ESN and then
gated into that same layer of the GRU. I would also like to try this technique with LSTMs as they normally perform better on stock data to begin with and since the hybrid model seemed to perform better than just the GRU.
I would also like to compare the hybrid model to more models like LSTMs and Transformer models.